In [ ]:
import pandas as pd
import re

print("Memulai proses penyatuan dan pembersihan dataset...")

# 1. Tentukan Path
path_lokasi_revisi = "../../data/dataset_tiket_lengkap_lokasi_revisi.csv"
path_revisi        = "../../data/dataset_tiket_lengkap_dupe_revisi.csv"
path_utama         = "../../data/dataset_tiket_lengkap_gedung.csv"

# (Asumsi nama kolom teks Anda adalah 'teks_keluhan' dan kolom labelnya 'LOKASI_GEDUNG')
# 2. Baca Semua Dataset
# Tambahkan on_bad_lines='skip' agar Pandas otomatis membuang baris yang cacat
df1 = pd.read_csv(path_lokasi_revisi, sep='|', engine='python', on_bad_lines='skip')
df2 = pd.read_csv(path_revisi, sep='|', engine='python', on_bad_lines='skip')
df3 = pd.read_csv(path_utama, sep='|', engine='python', on_bad_lines='skip')

# 3. Gabungkan (Concat) Menjadi Satu DataFrame
df_merged = pd.concat([df1, df2, df3], ignore_index=True)
print(f"Total baris setelah digabung: {len(df_merged)}")

# 4. Hapus Duplikat Identik
# Jika ada laporan yang sama persis akibat penggabungan, kita buang agar model tidak overfitting
df_merged = df_merged.drop_duplicates(subset=['teks_keluhan_awam']).reset_index(drop=True)
print(f"Total baris setelah hapus duplikat teks: {len(df_merged)}")

# 5. Hapus "Label Halusinasi LLM" (Masalah yang baru kita temukan tadi)
kondisi_tanpa_kata_gedung = ~df_merged['teks_keluhan_awam'].str.contains(r'\b(gedung|gdng|gd\.?|tower|twr|blok|blk|lobby|lobi|area)\b', case=False, na=False)
kondisi_label_terisi = ~df_merged['lokasi_gedung'].isin(['Unknown', 'Tidak Disebutkan', '-'])
baris_halusinasi = kondisi_tanpa_kata_gedung & kondisi_label_terisi

df_clean = df_merged[~baris_halusinasi].reset_index(drop=True)
print(f"Dihapus {baris_halusinasi.sum()} baris halusinasi LLM.")
print(f"Total baris FINAL yang siap masuk training: {len(df_clean)}")

# 6. Simpan ke dalam 1 File Utama (Golden Dataset)
path_final = "../../data/dataset_tiket_master_bersih.csv"
df_clean.to_csv(path_final, sep='|', index=False)

print(f"✅ Sukses! File utama berhasil disimpan di: {path_final}")

Memulai proses penyatuan dan pembersihan dataset...
Total baris setelah digabung: 2803
Total baris setelah hapus duplikat teks: 2360
Dihapus 473 baris halusinasi LLM.
Total baris FINAL yang siap masuk training: 1887
✅ Sukses! File utama berhasil disimpan di: ../../data/dataset_tiket_master_bersih.csv


C:\Users\ryama\AppData\Local\Temp\ipykernel_2220\2044302308.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  kondisi_tanpa_kata_gedung = ~df_merged['teks_keluhan_awam'].str.contains(r'\b(gedung|gdng|gd\.?|tower|twr|blok|blk|lobby|lobi|area)\b', case=False, na=False)


: 